# 4.3 Agent-Based Autonomous Flight -- Indoor Navigation

## New Scene

In this lesson, we use an indoor scene:

<img src='img/room_in.jpg' width='640px' />

Because indoor drone flight requires a smaller environment, we use a custom compiled version:

Download: https://pan.baidu.com/s/1mvBZOpPGSSYG5Iqv7sQduA?pwd=3pkn Code: 3pkn -- V6 version

The drone's new tasks include indoor navigation and searching for targets within the room.

## Tool Functions

We need to define SmolAgents tool functions. See airsim_smol_wrapper.py for the complete implementation:

```python

from smolagents import tool
import math
from typing import Optional, Tuple



@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0, # velocity
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781 N, 87.6298 W) to Sydney (33.8688 S, 151.2093 E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Convert coordinates to radians
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth radius (for Haversine formula)
    EARTH_RADIUS_KM = 6371.0

    # Use Haversine formula to calculate great-circle distance
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% for routing and air traffic
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format result
    return round(flight_time, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))

```



The function comment format follows **Google Style Docstrings**, which is one of the most widely used documentation string formats in Python (others include Numpy and reStructuredText styles). Here's the analysis:

---

### **Format Analysis**
1. **Section Headers**
- Uses section keywords (`Args`, `Returns`, `Example`) to organize content
- Follows Google style with colons after section names (`Args:`)

2. **Parameter and Return Value Documentation**
- Each parameter in the list includes **type** and **description** (e.g., `origin_coords: Tuple[float, float]`)
- Return value documentation includes type and description (e.g., `float: The estimated travel time in hours`)

3. **Examples**
- Provides usage examples through the `Example` section, using `>>>` to represent command-line input

4. **Type Annotations**
- Uses Python type hints (e.g., `Tuple[float, float]`) for parameter types
- Optional parameters use `Optional` with default values (e.g., `cruising_speed_kmh: Optional[float] = 750.0`)

This comment format is essential for tool definitions, as it provides the LLM with the API documentation it needs to correctly invoke the tools.

See the complete implementation in airsim_smol_wrapper.py

In [ ]:
#!pip install smolagents[litellm]
from smolagents import CodeAgent, LiteLLMModel

model = LiteLLMModel(
    model_id="openai/moonshotai/Kimi-K2.6",  # This model is a bit weak for agentic behaviours though
    api_base="https://api.intelligence.io.solutions/api/v1", # replace with 127.0.0.1:11434 or remote open-ai compatible server if necessary
    api_key="xxxxxxxxxxxxxxxxxxxxxxxxxxxx", # Replace with your own API key
    flatten_messages_as_text=True, # Some models may not support multi-step conversations well
)

In [ ]:
#import litellm
#litellm._turn_on_debug()

In [ ]:
from airsim_smol_wrapper import *
refer_info = """
Imagine you are helping me interact with the AirSim simulator.

We are controlling a physical Agent. At any given point in time, you have the following capabilities. You need to output code for certain requests.
- Ask me a clarification question.
- Explain why you did it that way.
- Output code commands that achieve the desired goal.

You must not use any other hypothetical functions. You can use functions from Python libraries such as math, numpy, etc. Are you ready?
"""

In [ ]:
agent = CodeAgent(tools=[takeoff,land], model=model)

In [ ]:
agent.run(
    "Take off the drone",
    additional_args={"refer_info":refer_info}
)

In [ ]:
takeoff()

In [ ]:
agent = CodeAgent(tools=[takeoff,get_drone_position, fly_to, fly_path, set_yaw, get_yaw, get_position, look_at,
                        turn_left, turn_right, forward, detect], model=model)

In [ ]:
command = """
The drone has taken off. Now I need it to explore the room. It can detect objects in the scene. Turn about 7 times to look around. This is an exploration task.
"""

result1 = agent.run(
    command
    #additional_args={"refer_info":refer_info}
)

In [ ]:
result1

In [ ]:
obj_id_list, obj_locs,img_with_box = detect_with_img("doorway")

In [ ]:
img_with_box

In [ ]:
#turn_right()
#set_yaw(0)
#reset()
#get_yaw()

In [ ]:
command = """
Good, now I need the drone to move forward 5 meters and navigate through the room.
"""

agent.run(
    command
)

In [ ]:
agent = CodeAgent(tools=[takeoff,get_drone_position, fly_to, fly_path, set_yaw, get_yaw, get_position, look_at,
                        turn_left, turn_right, forward, look], model=model
                  ) 

In [ ]:
command = """
Good, the drone is now inside the room. Can you look around and tell me what objects are in the room?
"""

result1 = agent.run(
    command
)